# Network Intrusion Detection — Pipeline Walkthrough

This notebook narrates the project pipeline end to end: **load → preprocess →
feature selection → classical baselines → deep learning → evaluation with
emphasis on the false-positive rate**.

> **Honest framing.** The original 2024 research code is no longer available,
> so this repository is a clean reference implementation of the research
> approach. This notebook runs on **synthetic data** (no download needed), so
> the numbers below are illustrative — they demonstrate the pipeline, not
> real IDS performance. The research outcomes (+27% accuracy over baseline,
> −18% false-positive rate with sensitivity held) are summarised at the end
> for context.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Make `src/` importable whether the notebook runs from the repo root or notebooks/
_here = Path().resolve()
_root = _here.parent if _here.name == "notebooks" else _here
sys.path.insert(0, str(_root / "src"))

from data_loader import generate_synthetic_ids_data, split_features_target
from preprocessing import TabularPreprocessor, handle_imbalance
from feature_selection import select_features
from models import (
    get_baseline_models,
    IntrusionMLP,
    make_dataloaders,
    train_torch_model,
    evaluate_torch,
)
from evaluate import binary_metrics, summarize_results, plot_confusion_matrix

print("imports ok")

## 2. Data — synthetic demo

`generate_synthetic_ids_data` builds a small IDS-like table (numeric flow
statistics plus categorical protocol/service fields) with the same tidy
schema the real loaders return. Swap this cell for `load_dataset(...)`
once you have downloaded UNSW-NB15 or NSL-KDD (see README).

In [ ]:
df = generate_synthetic_ids_data(n_samples=4000, random_state=42)
print("shape:", df.shape)
print(df["label"].value_counts(normalize=True).round(3))
df.head()

## 3. Preprocessing

One-hot encode categoricals, standardise numerics — every statistic is fit
on the training split only, then applied to the test split (no leakage).

In [ ]:
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pre = TabularPreprocessor(random_state=42)
X_train_p = pre.fit_transform(X_train)
X_test_p = pre.transform(X_test)
print(f"raw features: {X_train.shape[1]} -> encoded: {X_train_p.shape[1]}")
print("attack rate:", round(float(y_train.mean()), 3))

## 4. Feature selection

Isolate the signals that separate attacks from normal traffic *before*
benchmarking models — the step that lifted accuracy over the all-features
baseline in the original research. Here: top-12 by mutual information.

In [ ]:
sel = select_features(X_train_p, y_train, method="mutual_info", k=12)
print(f"selected {len(sel.selected_features)} features:")
print(sel.selected_features)
X_train_s = sel.X_selected
X_test_s = X_test_p[sel.selected_features]

## 5. Classical baselines — Random Forest, XGBoost, SVM

In [ ]:
results, all_preds = {}, {}
for name, clf in get_baseline_models(random_state=42).items():
    clf.fit(X_train_s, y_train)
    y_pred = clf.predict(X_test_s)
    results[name] = binary_metrics(y_test, y_pred)
    all_preds[name] = y_pred
    print(f"{name:14s} acc={results[name]['accuracy']:.4f} "
          f"fpr={results[name]['fpr']:.4f}")

summarize_results(results).round(4)

## 6. Deep learning — PyTorch MLP

The deep-learning stage of the research is what cut the false-positive
rate while holding detection sensitivity. A small MLP trained for a few
epochs is enough to demonstrate the workflow here.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_s, y_train, test_size=0.15, random_state=42, stratify=y_train
)
train_loader, val_loader = make_dataloaders(
    X_tr.values, y_tr.values, X_val.values, y_val.values, batch_size=256
)

mlp = IntrusionMLP(input_dim=X_train_s.shape[1], hidden_dims=(64, 32), dropout=0.2)
history = train_torch_model(mlp, train_loader, val_loader, epochs=10, verbose=True)

test_loader, _ = make_dataloaders(
    X_test_s.values, y_test.values, batch_size=512, shuffle_train=False
)
ev = evaluate_torch(mlp, test_loader)
results["mlp"] = binary_metrics(ev["y_true"], ev["y_pred"])
all_preds["mlp"] = ev["y_pred"]

summarize_results(results).round(4)

## 7. Results — why the false-positive rate matters

An IDS that cries wolf burns out the SOC: every false alarm costs analyst
time, and chronic noise trains people to ignore real alerts. Accuracy can
hide this, so we pick the winner by **lowest FPR at acceptable recall**.

In [ ]:
best = min(results, key=lambda n: results[n]["fpr"])
print(f"lowest false-positive rate: {best}  "
      f"(FPR = {results[best]['fpr']:.4f}, recall = {results[best]['recall']:.4f})")
fig = plot_confusion_matrix(y_test, all_preds[best])
fig

## 8. Running on the real benchmarks

1. Download the datasets (see README for links):
   - `UNSW_NB15_training-set.csv` / `UNSW_NB15_testing-set.csv`, or
   - `KDDTrain+.txt` / `KDDTest+.txt`
   into `data/`.
2. Run the full pipeline:
   ```bash
   python src/train.py --dataset nsl-kdd --data-dir data --model all \
       --feature-selection mutual_info --k 25
   ```

### Research outcomes (original 2024 study, for context)

| Outcome | Detail |
|---|---|
| **+27% detection accuracy** | over the baseline via systematic feature selection and model benchmarking |
| **−18% false-positive rate** | via deep learning, with detection sensitivity maintained |
| **Publication** | *"Intrusion Detection System,"* University at Albany Workshop, December 2024 |

These figures describe the original study — re-running this
reimplementation on the real benchmarks will produce its own numbers.